# 기존 gzip coverage 밖 확장 — C CPU pilot

기존 A/B를 다시 실행하지 않습니다. 한 번의 새 CPU runtime C에서 고정 B anchor/snapshot을 복원하고, 기존 coverage 밖까지16member씩 최대8회만 탐색합니다. 새 restart point 뒤의 완전 JSON을 선택해 C snapshot을 저장한 뒤, 같은 runtime의 **새 로컬 DB+새 gzip handle**로 복원하여 자동index생성없이cold 추출합니다. C 뒤에 VM삭제/추가runtime 반복은 요구하지 않습니다.

Source/manifest/B-anchor SHA와 revision은 검토된 ready에 고정합니다. 입력 B폴더와 새 출력 C폴더는 분리합니다. 한 Runtime의 합계8MiB 응답body/120HTTP/300초와 phase합read32MiB/decoded32MiB/전체ledger300초를 적용합니다. Auth와 최초 B복원은 ledger전이고, C snapshot복사/복원은각자60초이며 더엄격한전체ledger에도포함됩니다. 원본budgetclock은재설정하지않습니다.

파일 수만 증가하거나 기존 coverage와 같으면 통과하지 않습니다. coverage/boundary 전진, 옛catalog보존, 신규restartpoint 실사용과 기존/새 JSON hash검증, 부모B/lineage가맞는 C 저장본재복원이모두필요합니다. 상한/EOF/미달은partial,변경/게시후재검증실패는failed로보존하며 자동반복·예산확대는하지않습니다.


In [ ]:
# Root 검토·커밋 후 만든 C ready notebook에 pin을 채웁니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
MANIFEST_SHA256 = ""
PHASE = "C"
RUN_ID = "pose_persistent_extension_001"
EXPECTED_ANCHOR_SHA256 = ""  # 기존 B anchor를 독립 보관한 정확한 SHA
INPUT_SNAPSHOT_DIRECTORY = "/content/drive/MyDrive/reports/persistent-index/pose_persistent_86d322c_001"
OUTPUT_SNAPSHOT_DIRECTORY = "/content/drive/MyDrive/reports/persistent-index/pose_persistent_extension_001"
WORKSPACE = "/content/rewind-index-extension"


## 1. 고정 코드 ZIP 업로드

첫 업로드에는 source ZIP 한 개만 선택합니다. 실행 결과 ZIP·snapshot ZIP을 올리면 거부합니다. Source SHA/revision은 ready에 고정돼 있어야 합니다.


In [ ]:
import sys, os, io, stat, json, tempfile, zipfile, shutil
from pathlib import Path, PurePosixPath
from google.colab import files

# 순차 해제 누적량과 RAM 사용량은 다릅니다. 업로드/런타임 여유를 먼저 확인합니다.
meminfo=Path("/proc/meminfo").read_text() if Path("/proc/meminfo").is_file() else ""
available_ram=next((int(row.split()[1])*1024 for row in meminfo.splitlines() if row.startswith("MemAvailable:")),None)
free_disk=shutil.disk_usage(".").free
print({"python":sys.version.split()[0],"free_disk_bytes":free_disk,"available_ram_bytes":available_ram})
if free_disk<2*1024**3 or available_ram is None or available_ram<2*1024**3:
    raise RuntimeError("가용 RAM과 임시 디스크가 각각2GiB 이상인 Colab CPU 런타임을 준비하세요.")
uploaded = files.upload()  # 여기에는 코드가 들어 있는 source ZIP 한 개만 올립니다.
if len(uploaded)!=1:
    raise ValueError("코드 source ZIP 한 개만 선택하세요.")
source_name, source_bytes = next(iter(uploaded.items()))
import hashlib
if len(SOURCE_ZIP_SHA256)!=64 or hashlib.sha256(source_bytes).hexdigest()!=SOURCE_ZIP_SHA256:
    raise ValueError("고정 source ZIP SHA256 불일치. 결과 ZIP을 올리지 마세요.")
if not source_name.lower().endswith(".zip") or len(source_bytes)>64*1024**2:
    raise ValueError("64MiB 이하의 코드 source ZIP이 필요합니다.")
excluded={".git",".venv","data","outputs","checkpoints",".aws",".codex",".agents","__pycache__"}
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    members=archive.infolist()
    if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
        raise ValueError("코드 ZIP의 항목 수/해제 크기 상한을 초과했습니다.")
    code_entries=[m for m in members if m.filename.endswith("rewind_archive_index/__init__.py")]
    if len(code_entries)!=1:
        inner=[PurePosixPath(m.filename).name for m in members if m.filename.lower().endswith(".zip")][:6]
        if inner:
            raise ValueError("배포 묶음 ZIP입니다. 로컬에서 내부 코드 ZIP "+repr(inner)+"만 꺼내 다시 올리세요. 중첩 자동 해제는 하지 않습니다.")
        raise ValueError("rewind_archive_index/__init__.py를 포함한 코드 source ZIP이 필요합니다. Notebook/결과 ZIP과 구분하세요.")
    seen=set()
    for m in members:
        p=PurePosixPath(m.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in m.filename or any(":" in x for x in p.parts) or any(ord(c)<32 for c in m.filename) or stat.S_ISLNK(m.external_attr>>16):
            raise ValueError("안전하지 않은 코드 ZIP 경로입니다.")
        key=p.as_posix().casefold().rstrip("/")
        if key in seen:raise ValueError("중복 코드 ZIP 경로입니다.")
        seen.add(key)
    extract_root=Path(tempfile.mkdtemp(prefix="rewind_source_",dir="/content"))
    for m in members:
        parts=PurePosixPath(m.filename).parts
        if not any(p in excluded or p==".env" or p.startswith(".env.") for p in parts):
            archive.extract(m,extract_root)
roots=[p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots)!=1:raise ValueError("단일 프로젝트 source ZIP이 필요합니다.")
PROJECT_ROOT=roots[0]
if not (PROJECT_ROOT/"rewind_archive/workflow.py").is_file():
    raise ValueError("이 notebook과 맞는 최신 source ZIP이 필요합니다(workflow.py 누락).")
if any(name=="rewind_archive" or name.startswith("rewind_archive.") or name.startswith("rewind_archive_index") for name in sys.modules):
    raise RuntimeError("이전 모듈이 이미 import되어 있습니다. 기존 결과를 먼저 보관하고 런타임을 재시작한 뒤 새 source ZIP으로 시작하세요.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))
from rewind_archive.workflow import NOTEBOOK_API_VERSION
if NOTEBOOK_API_VERSION!=10:raise RuntimeError("Notebook/source API 버전이 다릅니다.")
stamp=PROJECT_ROOT/"SOURCE_REVISION.txt"
print({"source_root":str(PROJECT_ROOT),"source_revision":stamp.read_text().strip() if stamp.exists() else "unversioned; code hashes recorded","notebook_api":NOTEBOOK_API_VERSION})
del uploaded, source_bytes
if not stamp.is_file() or not SOURCE_REVISION or stamp.read_text().strip()!=SOURCE_REVISION:
    raise ValueError("고정 source revision 불일치")
from rewind_archive_index.extension_pilot import run_extension, PILOT_DRIVE_LIMITS
print({"persistent_drive_entrypoint":True,"fixed_phase_limits":vars(PILOT_DRIVE_LIMITS)})


## 2. 선택 의존성

CPU 런타임에서 명시적으로 한 번 설치합니다. A/B 모두 같은 indexed_gzip1.10.3을 사용합니다. 모델·torch 설치는 필요 없습니다.


In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.run([sys.executable,"-m","pip","install","indexed_gzip==1.10.3","psutil>=5.9,<8"],check=True)


## 3. 고정 metadata와 신뢰 anchor

metadata에는 모든 원본 조각의 ID/크기/version/수정시각이 있어야 하며 source ZIP과는 별도 파일입니다. 개인 ID는 notebook에 하드코딩하지 않습니다. C에서는 기존 B에서 내려받아 별도로 보관한 anchor 파일의 SHA를 먼저 대조합니다. Drive에 같이 놓인 파일만으로 최신/분기 신뢰를 결정하지 않습니다.


In [ ]:
import hashlib, json
from rewind_archive_index.drive import make_drive_source
manifest_upload=files.upload()
if len(manifest_upload)!=1:raise ValueError("고정 archive_manifest.json 한 개를 선택하세요.")
manifest_bytes=next(iter(manifest_upload.values()))
if len(manifest_bytes)>128*1024 or len(MANIFEST_SHA256)!=64 or hashlib.sha256(manifest_bytes).hexdigest()!=MANIFEST_SHA256:
    raise ValueError("manifest 크기/SHA 불일치")
MANIFEST=json.loads(manifest_bytes)
make_drive_source(MANIFEST)
ANCHOR=None
if PHASE=="C":
    anchor_upload=files.upload()
    if len(anchor_upload)!=1:raise ValueError("고정 B anchor JSON 한 개를 선택하세요.")
    anchor_bytes=next(iter(anchor_upload.values()))
    if len(anchor_bytes)>65536 or len(EXPECTED_ANCHOR_SHA256)!=64 or hashlib.sha256(anchor_bytes).hexdigest()!=EXPECTED_ANCHOR_SHA256:
        raise ValueError("신뢰 anchor SHA 불일치")
    ANCHOR=json.loads(anchor_bytes)
print({"phase":PHASE,"run_id":RUN_ID,"metadata_checked_offline":True,"anchor_pinned":ANCHOR is not None})


## 4. Drive mount

Live SQLite는 `/content`의 로컬 workspace에서만 실행합니다. 입력 B폴더는 읽기만하고 새 C폴더에 닫힌 ZIP과 receipt만 저장합니다. 폴더의 최신 파일명을 자동으로 고르지 않습니다.


In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## 5. 한 번 실행

설정·phase·A anchor·예산을 테스트 세션과 확인한 뒤 gate만 켭니다. 시작/실패/성공 marker가 남은 같은 run_id/phase는 재실행을 거부합니다. 실패해도 보고서를 보존하고 예산을 바꾸거나 자동 재시도하지 않습니다.


In [ ]:
EXECUTE_PHASE = False
def create_drive_session():
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession, Request
    from requests.adapters import HTTPAdapter
    import urllib3
    if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
        raise RuntimeError("urllib3 raw.read1 API가 필요합니다.")
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    # opt-in에서는 인증용 HTTP session도 숨은 retry0으로 고정한다.
    auth_http = None
    if True:
        import requests
        auth_http=requests.Session()
        auth_http.trust_env=False
        auth_http.mount("https://",HTTPAdapter(max_retries=0))
    try:
        auth_request=Request(session=auth_http) if auth_http is not None else Request()
        credentials.refresh(auth_request)
        session=AuthorizedSession(credentials,max_refresh_attempts=0,auth_request=auth_request) if auth_http is not None else AuthorizedSession(credentials,max_refresh_attempts=0)
        if auth_http is not None:
            # custom auth_request session은 AuthorizedSession.close가 소유하지 않는다.
            original_close=session.close
            def close_sessions():
                try:original_close()
                finally:auth_http.close()
            session.close=close_sessions
        session.trust_env=False
        session.mount("https://",HTTPAdapter(max_retries=0))
    except BaseException:
        if auth_http is not None:auth_http.close()
        raise
    return session

if EXECUTE_PHASE:
    mount=Path("/content/drive")
    if not os.path.ismount(mount) or any(not Path(p).resolve().is_relative_to(mount.resolve()) for p in (INPUT_SNAPSHOT_DIRECTORY,OUTPUT_SNAPSHOT_DIRECTORY)):
        raise RuntimeError("Drive가 실제 mount됐고 snapshot 경로가 그 하위인지 먼저 확인하세요. 인증/marker를 시작하지 않았습니다.")
    if Path(INPUT_SNAPSHOT_DIRECTORY).resolve()==Path(OUTPUT_SNAPSHOT_DIRECTORY).resolve():raise RuntimeError("입력B/출력C 폴더를 분리하세요.")
    REPORT=run_extension(run_id=RUN_ID,workspace=WORKSPACE,input_snapshot_directory=INPUT_SNAPSHOT_DIRECTORY,
        output_snapshot_directory=OUTPUT_SNAPSHOT_DIRECTORY,manifest=MANIFEST,session_factory=create_drive_session,anchor=ANCHOR)
    import importlib.metadata
    def installed_version(name):
        try:return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:return None
    REPORT["execution_metadata"]={"source_revision":SOURCE_REVISION,"source_zip_sha256":SOURCE_ZIP_SHA256,
        "manifest_sha256":MANIFEST_SHA256,"python":sys.version,"indexed_gzip":installed_version("indexed_gzip"),"psutil":installed_version("psutil")}
    (Path(WORKSPACE)/(RUN_ID+"-"+PHASE+".json")).write_text(json.dumps(REPORT,ensure_ascii=False,indent=2),encoding="utf-8")
    print(json.dumps(REPORT,ensure_ascii=False,indent=2))
else:
    print("인증·원본 읽기·snapshot 게시를 실행하지 않았습니다.")


## 결과 보존·선택 flush

passed/partial/failed 모두 C보고서 ZIP을 보관합니다. 게시한 C의재검증이실패해도snapshot/receipt/anchor 경로를보고서에남깁니다. partial이면 Csnapshot이없을수있습니다. 기존B는덮어쓰지않습니다.

첫 실행은 `DOWNLOAD_REPORT=True`, `FLUSH_AND_UNMOUNT=False`로내려받습니다. 저장확인후 `DOWNLOAD_REPORT=False`, `FLUSH_AND_UNMOUNT=True`로선택flush(timeout_ms60000)를할수있습니다. 실패시현재runtime과보고서를보존합니다. **C완료뒤runtime를삭제하고또실험하라고요구하지않습니다.** 새로컬DB검증은동일C안에서수행하며전역최신판정/새VMDrive내구성재시험은아닙니다.


In [ ]:
DOWNLOAD_REPORT = False
FLUSH_AND_UNMOUNT = False
if DOWNLOAD_REPORT:
    import zipfile, hashlib
    if "REPORT" not in globals() or REPORT.get("phase")!=PHASE or REPORT.get("run_id")!=RUN_ID:
        raise RuntimeError("현재 phase의 보고서가 필요합니다.")
    work=Path(WORKSPACE)
    paths=[work/(RUN_ID+"-"+PHASE+".json")]
    for suffix in ("-anchor.json","-flush.json"):
        path=work/(RUN_ID+"-"+PHASE+suffix)
        if path.is_file():paths.append(path)
    for row in REPORT.get("extraction",{}).get("files",[]):
        path=work/(RUN_ID+"-"+PHASE+"-selected")/row["relative_path"]
        if path.is_symlink() or not path.resolve().is_relative_to(work.resolve()):raise RuntimeError("보고서 경로 오류")
        paths.append(path)
    if len(paths)>8 or sum(path.stat().st_size for path in paths)>20*1024**2:raise RuntimeError("보고서 ZIP 상한")
    REPORT_ZIP=work/(RUN_ID+"-"+PHASE+"-report.zip")
    if REPORT_ZIP.exists():raise RuntimeError("보고서 ZIP이 이미 있습니다. 기존 파일을 다시 내려받으세요.")
    with zipfile.ZipFile(REPORT_ZIP,"x",zipfile.ZIP_STORED) as z:
        for path in paths:z.write(path,path.relative_to(work).as_posix())
    print({"report_zip":str(REPORT_ZIP),"sha256":hashlib.sha256(REPORT_ZIP.read_bytes()).hexdigest()})
    files.download(str(REPORT_ZIP));REPORT_BUNDLE_REQUESTED=True
if FLUSH_AND_UNMOUNT:
    if not globals().get("REPORT_BUNDLE_REQUESTED") or REPORT.get("status")!="passed":
        raise RuntimeError("성공 보고서/anchor ZIP을 먼저 내려받아 보관을 확인하세요. 실패 runtime은 보존하세요.")
    from google.colab import drive
    try:
        drive.flush_and_unmount(timeout_ms=60000)
        FLUSH_REPORT={"flush":"returned","timeout_ms":60000,"persistence_proven":False}
    except Exception:
        FLUSH_REPORT={"flush":"failed","timeout_ms":60000,"persistence_proven":False,"action":"현재 런타임과 보고서를 보존하고 테스트 세션에 전달"}
    FLUSH_PATH=Path(WORKSPACE)/(RUN_ID+"-"+PHASE+"-flush.json")
    FLUSH_PATH.write_text(json.dumps(FLUSH_REPORT,ensure_ascii=False))
    print(FLUSH_REPORT);files.download(str(FLUSH_PATH))
